# 产业原材料智能选型 Skill - Demo

将自然语言需求转化为数据驱动的材料推荐、排序、风险分析与验证规划。

## 1. 安装依赖（如尚未安装）

In [ ]:
# 如果首次运行，取消注释以下行安装依赖
# !pip install -r requirements.txt

## 2. 加载 Pipeline

In [ ]:
import sys
import os
import json

# 切换到项目根目录
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pipeline import MaterialSelectionPipeline
from src.retriever import MaterialRetriever
from src.ranking import TopsisRanker

print("模块加载成功")
print(f"项目路径: {project_root}")

## 3. 查看材料数据库概览

In [ ]:
retriever = MaterialRetriever()
df = retriever.get_all_materials()
print(f"数据库共 {len(df)} 条材料记录\n")
print("材料牌号一览:")
for _, row in df.iterrows():
    print(f"  {row['material_id']:12s} | {row['grade']:20s} | {row['category']:10s} | 密度={row['density']} g/cm³ | 拉伸={row['tensile_strength']} MPa | 成本={row['estimated_cost_per_kg']} $/kg")

## 4. 运行完整选型 Pipeline

In [ ]:
# 初始化 Pipeline（无 API Key 时自动使用离线模式）
pipeline = MaterialSelectionPipeline()

# 典型输入：电动汽车电池包壳体选材
user_input = (
    "电动汽车电池包壳体，"
    "密度<2.0 g/cm³，"
    "拉伸强度>300 MPa，"
    "导热系数>100 W/mK，"
    "UL94 V-0，"
    "成本<$20/kg，"
    "RoHS合规，"
    "国内采购，"
    "年用量1000吨"
)

print("=" * 60)
print("用户输入:")
print(user_input)
print("=" * 60)

# 运行 Pipeline
result = pipeline.run(user_input)

print("\n" + "=" * 60)
print("Pipeline 运行完成!")
print("=" * 60)

## 5. 查看 JSON 输出摘要

In [ ]:
summary = result.get('summary', {})
print(f"应用场景: {result.get('application', '')}")
print(f"候选材料数: {summary.get('total_candidates', 0)}")
print(f"否决材料数: {summary.get('vetoed_count', 0)}")
print(f"首选推荐: {summary.get('top_recommendation', '无')}")
print(f"推荐置信度: {summary.get('recommendation_confidence', '中')}")

print("\n--- 候选材料排序 ---")
for m in result.get('ranked_materials', []):
    props = m['key_properties']
    print(f"  #{m['rank']} {m['grade']:20s} | {m['category']:8s} | "
          f"密度={props['density']} | 拉伸={props['tensile_strength']} | "
          f"导热={props['thermal_conductivity']} | "
          f"阻燃={props['flammability_UL94']} | "
          f"成本={props['cost_per_kg']} | "
          f"TOPSIS={m['topsis_score']:.4f}")

print("\n--- 否决材料 ---")
for v in result.get('veto_details', []):
    print(f"  {v['grade']:20s} | {v['veto_reason']}")

print("\n--- 风险分析 ---")
for r in result.get('risk_analysis', []):
    print(f"  {r['grade']:20s} | 风险等级: {r['overall_risk_level']} | "
          f"工艺风险: {len(r['process_risks'])} | "
          f"供应链风险: {len(r['supply_risks'])} | "
          f"数据质量风险: {len(r['data_quality_risks'])}")

print("\n--- 验证计划 ---")
for step in result.get('next_steps', []):
    print(f"  - {step}")

## 6. 生成 Markdown 报告

In [ ]:
from src.reporter import ReportGenerator

reporter = ReportGenerator()
markdown_report = reporter.generate_markdown_report(result)

# 保存到文件
report_path = os.path.join(project_root, 'selection_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(markdown_report)

print(f"Markdown 报告已保存至: {report_path}")
print("\n" + "=" * 60)
print(markdown_report[:2000] + "\n...（报告完整内容已保存至文件）")

## 7. 雷达图可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_radar(radar_data, title="材料性能雷达图"):
    """绘制雷达图"""
    if not radar_data:
        print("无雷达图数据")
        return
    
    # 提取属性名
    props = list(radar_data[0]["values"].keys())
    num_vars = len(props)
    
    # 计算角度
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]  # 闭合
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    
    for i, entry in enumerate(radar_data):
        values = [entry["values"].get(p, 0) for p in props]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=entry["grade"], color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(props, fontsize=10)
    ax.set_title(title, size=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    
    plt.tight_layout()
    plt.show()

# 绘制雷达图
radar_data = result.get('radar_chart_data', [])
if radar_data:
    plot_radar(radar_data)
else:
    print("当前结果无雷达图数据（可能所有材料均被否决）")
    print("\n查看原始数据中的雷达图字段:")
    print(json.dumps(radar_data, ensure_ascii=False, indent=2))

## 8. 探索其他场景

In [ ]:
def run_scenario(description, user_input):
    """运行一个场景并打印摘要"""
    print(f"\n{'=' * 60}")
    print(f"场景: {description}")
    print(f"输入: {user_input}")
    print('=' * 60)
    
    result = pipeline.run(user_input)
    summary = result.get('summary', {})
    print(f"候选材料: {summary.get('total_candidates', 0)} | 否决: {summary.get('vetoed_count', 0)}")
    print(f"首选推荐: {summary.get('top_recommendation', '无')} (置信度: {summary.get('recommendation_confidence', '中')})")
    
    # 显示 Top 3
    for m in result.get('ranked_materials', [])[:3]:
        print(f"  #{m['rank']} {m['grade']:20s} TOPSIS={m['topsis_score']:.4f}")
    
    return result

# 场景1：汽车发动机罩盖（轻量化 + 阻燃）
r1 = run_scenario(
    "汽车发动机罩盖 - 轻量化阻燃",
    "汽车发动机罩盖，密度<1.5 g/cm³，拉伸强度>150 MPa，UL94 V-0，成本<$10/kg，RoHS合规"
)

# 场景2：散热器基板（高导热优先）
r2 = run_scenario(
    "散热器基板 - 高导热优先",
    "散热器基板，导热系数>150 W/mK，密度<3.0 g/cm³，成本<$50/kg"
)

# 场景3：高强度结构件
r3 = run_scenario(
    "高强度结构件 - 强度优先",
    "高强度结构件，拉伸强度>500 MPa，屈服强度>400 MPa，密度<5.0 g/cm³，成本<$30/kg"
)

## 9. 查看完整 JSON 输出

In [ ]:
# 输出完整 JSON 结构
print(json.dumps(result, ensure_ascii=False, indent=2)[:5000])
print("\n...（输出截断，完整 JSON 结构见上方）")

## 10. 推理溯源

In [ ]:
trace = result.get('trace', {})
print("推理溯源信息:")
print(f"  检索方式: {trace.get('retrieval_method', '')}")
print(f"  TDS 条目数: {trace.get('tds_entries', 0)}")
print(f"  排序算法: {trace.get('sorting_algorithm', '')}")
print("\n处理步骤:")
for step in trace.get('steps', []):
    print(f"  步骤{step.get('step', '')}: {step.get('action', '')}")